In [3]:
import os, sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch
    print("PyTorch:", torch.__version__)    
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch is not installed yet.")

Python: 3.12.15 (main, Oct  9 2026, 15:22:32) [Clang 22.1.3 ]
Platform: Linux-6.18.56-1-lts-x86_64-with-glibc2.44
PyTorch: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4080 Laptop GPU


In [2]:
import shutil, subprocess

flyvis_cli = shutil.which("flyvis")
if flyvis_cli is None:
    raise RuntimeError("The flyvis command was not found. Restart the notebook kernel after installation, then rerun this cell.")
result = subprocess.run([flyvis_cli, "download-pretrained"], text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
result.check_returncode()

Progress results_umap_and_clustering.zip: 100%
Checksum OK for results_umap_and_clustering.zip.
Unpacked results_umap_and_clustering.zip.
Progress results_pretrained_models.zip: 100%
Checksum OK for results_pretrained_models.zip.
Unpacked results_pretrained_models.zip.

[2026-10-14 04:53:54] font_manager:1263 Failed to extract font properties from /usr/share/fonts/noto/NotoColorEmoji.ttf: Non-scalable fonts are not supported
[2026-10-14 04:53:55] font_manager:1848 generated new fontManager
[2026-10-14 04:53:56] __init__:49 file_cache is only supported with oauth2client<4.0.0



In [1]:
import os
print("FLYVIS_ROOT_DIR =", os.environ.get("FLYVIS_ROOT_DIR", "(not set; FlyVis uses its default data directory)"))

try:
    import flyvis
    print("flyvis package:", flyvis.__file__)
except Exception as e:
    print("Could not import flyvis:", repr(e))

FLYVIS_ROOT_DIR = (not set; FlyVis uses its default data directory)


/home/bibu/flyvis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


flyvis package: /home/bibu/flyvis/lib/python3.12/site-packages/flyvis/__init__.py


In [4]:
from pathlib import Path
import os
import shutil

PROJECT_DIR = Path.home() / "Desktop/Project/Flyvis"
DATA_DIR = PROJECT_DIR / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Data:", DATA_DIR)

Project: /home/bibu/Desktop/Project/Flyvis
Data: /home/bibu/Desktop/Project/Flyvis/data


In [ ]:
import flyvis

PACKAGE_DATA = Path(flyvis.__file__).parent / "data"

for filename in [
    "responses_norm.h5",
    "results_pretrained_models.zip",
    "results_umap_and_clustering.zip",
]:
    source = PACKAGE_DATA / filename
    destination = DATA_DIR / filename

    if source.exists() and not destination.exists():
        shutil.copy2(source, destination)
        print("Copied:", filename)
    elif destination.exists():
        print("Already exists:", filename)
    else:
        print("Not found:", source)

In [7]:
from pathlib import Path

project_dir = Path.home() / "Desktop/Project/Flyvis"
project_dir.mkdir(parents=True, exist_ok=True)

env_file = project_dir / ".env"
env_text = "FLYVIS_ROOT_DIR=" + str(project_dir / "data") + "\n"

if env_file.exists():
    existing = env_file.read_text()
    lines = [
        line for line in existing.splitlines()
        if not line.strip().startswith("FLYVIS_ROOT_DIR=")
    ]
    existing = "\n".join(lines).strip()
    env_file.write_text((existing + "\n" if existing else "") + env_text)
else:
    env_file.write_text(env_text)

print(env_file.read_text())

FLYVIS_ROOT_DIR=/home/bibu/Desktop/Project/Flyvis/data



In [5]:
from pathlib import Path

old_dir = Path(flyvis.__file__).parent / "data"
new_dir = Path(flyvis.root_dir)

print("Old directory:", old_dir)
print("New directory:", new_dir)

print("\nOld files:")
for p in old_dir.iterdir():
    print(p.name)

print("\nNew files:")
for p in new_dir.iterdir():
    print(p.name)

Old directory: /home/bibu/flyvis/lib/python3.12/site-packages/flyvis/data
New directory: /home/bibu/Desktop/Project/Flyvis/data

Old files:
results_pretrained_models.zip
responses_norm.h5
results
results_umap_and_clustering.zip

New files:
results_pretrained_models.zip
responses_norm.h5
results_umap_and_clustering.zip


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import flyvis

data_dir = Path(flyvis.root_dir)

zip_files = [
    "results_pretrained_models.zip",
    "results_umap_and_clustering.zip",
]

for filename in zip_files:
    zip_path = data_dir / filename

    if not zip_path.exists():
        print(f"❌ ZIP not found: {zip_path}")
        continue

    print(f"\nExtracting {filename}...")

    with ZipFile(zip_path, "r") as z:
        # Protect against paths that would escape the data directory.
        root = data_dir.resolve()
        unsafe = [
            name for name in z.namelist()
            if not (data_dir / name).resolve().is_relative_to(root)
        ]
        if unsafe:
            raise ValueError(f"Unsafe ZIP paths detected: {unsafe[:5]}")

        # Extract only files that do not already exist.
        for member in z.infolist():
            if member.is_dir():
                (data_dir / member.filename).mkdir(
                    parents=True, exist_ok=True
                )
                continue

            destination = data_dir / member.filename
            if destination.exists():
                continue

            destination.parent.mkdir(parents=True, exist_ok=True)
            with z.open(member) as src, destination.open("wb") as dst:
                import shutil
                shutil.copyfileobj(src, dst)

    print(f"✅ Extracted {filename}")

print("\nExtraction complete.")
print("Data directory:", data_dir)

FILE  responses_norm.h5 (56.1 KB)
DIR   results/
DIR   results/flow/
DIR   results/flow/0000/
DIR   results/flow/0000/000/
FILE  results/flow/0000/000/_meta.yaml (2.2 KB)
FILE  results/flow/0000/000/best_chkpt (48.2 KB)
DIR   results/flow/0000/000/chkpts/
FILE  results/flow/0000/000/chkpts/chkpt_00000 (48.2 KB)
DIR   results/flow/0000/000/validation/
FILE  results/flow/0000/000/validation/loss.h5 (2.0 KB)
FILE  results/flow/0000/000/validation_loss.h5 (2.0 KB)
DIR   results/flow/0000/001/
FILE  results/flow/0000/001/_meta.yaml (2.2 KB)
FILE  results/flow/0000/001/best_chkpt (48.2 KB)
DIR   results/flow/0000/001/chkpts/
FILE  results/flow/0000/001/chkpts/chkpt_00000 (48.2 KB)
DIR   results/flow/0000/001/validation/
FILE  results/flow/0000/001/validation/loss.h5 (2.0 KB)
FILE  results/flow/0000/001/validation_loss.h5 (2.0 KB)
DIR   results/flow/0000/002/
FILE  results/flow/0000/002/_meta.yaml (2.2 KB)
FILE  results/flow/0000/002/best_chkpt (48.2 KB)
DIR   results/flow/0000/002/chkpts/
FI